# TabNet 적용

TabNet: 매 스텝마다 중요한 변수만 선택
Sparse learning 이라 중요 변수 몇 개만 집중해서 과적합을 줄인대 * 대신 데이터가 최소 수천이라는데

In [2]:
#!pip install pytorch-tabnet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

#!pip install catboost

In [4]:
df_train = pd.read_csv('TrainData/train.csv')
df_train = df_train.drop(['ID','generation'], axis=1)

df_test = pd.read_csv('OriginalData/test.csv')
df_test = df_test.drop(['ID','generation'], axis=1)

df_train.head()

,school1,major type,major1_1,major1_2,major_data,job,class1,class2,class3,class4,...,incumbents_company_level,incumbents_lecture_type,incumbents_lecture_scale,incumbents_lecture_scale_reason,interested_company,expected_domain,contest_participation,idea_contest,onedayclass_topic,completed
0,22,"복수 전공 ( 다중전공, 이중전공 포함 )",경제통상학,자연과학,False,대학생,1,4.0,NaN,NaN,...,해외 기업 (빅테크),"온, 오프라인 동시",100명 이상의 리스너와 10명 이상의 현직자,다양한 사람들과 만나서 생각을 교류할 수 있기 때문,"구글 딥마인드, 카카오 브레인","M. 전문, 과학 및 기술 서비스업",NaN,NaN,"Python 응용, 데이터 시각화 (Matplotlib, Seaborn 등), 머신...",0
1,1,"복수 전공 ( 다중전공, 이중전공 포함 )",자연과학,IT(컴퓨터 공학 포함),True,대학생,8,NaN,NaN,NaN,...,국내 빅테크 IT 계열 (네카라쿠배당토),오프라인,3~50명 내외의 강의 리스너와 1명의 현직자,더 많은 사람들이 있으면 제가 예상하지 못한 질문도 할 수 있다고 생각하기 때문입니다.,제일 기획,"J. 정보통신업, O. 공공 행정, 국방 및 사회보장 행정",NaN,NaN,머신러닝 / 딥러닝 응용,0
2,27,단일 전공,예체능,NaN,False,대학생,7,NaN,NaN,NaN,...,"국내 대기업 IT 계열 (금융, 제조 ...)",오프라인,3~50명 내외의 강의 리스너와 1명의 현직자,인원이 너무 적으면 서로 부담스러울 수 있을 것 같지만 너무 많으면 너무 피상적인 ...,Lg전자,"C. 제조업, K. 금융 및 보험업, R. 예술, 스포츠 및 여가관련 서비스업",NaN,NaN,"머신러닝 / 딥러닝 응용, SQL 응용, 웹 크롤링",0
3,1,"복수 전공 ( 다중전공, 이중전공 포함 )",사회과학,IT(컴퓨터 공학 포함),False,대학생,7,NaN,NaN,NaN,...,국내 빅테크 IT 계열 (네카라쿠배당토),"온, 오프라인 동시",3~50명 내외의 강의 리스너와 1명의 현직자,너무 많은 인원이 강의하면 루즈해질 것 같아서,네이버,"J. 정보통신업, K. 금융 및 보험업",NaN,NaN,머신러닝 / 딥러닝 응용,1
4,16,"복수 전공 ( 다중전공, 이중전공 포함 )",IT(컴퓨터 공학 포함),IT(컴퓨터 공학 포함),True,대학생,8,NaN,NaN,NaN,...,"국내 대기업 IT 계열 (금융, 제조 ...)","온, 오프라인 동시",3~50명 내외의 강의 리스너와 1명의 현직자,다양한 사람들에게 기회가 있으면 좋겠습니다.,네이버,"K. 금융 및 보험업, M. 전문, 과학 및 기술 서비스업, R. 예술, 스포츠 및...",NaN,NaN,"머신러닝 / 딥러닝 응용, SQL 응용, 웹 크롤링",0


In [5]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 유입경로 그룹화
    def route_group(x):
        x = str(x)
        if ('에브리타임' in x) or ('교내' in x):
            return '커뮤니티'
        if '인스타' in x:
            return 'SNS'
        if ('기존' in x) or ('오픈채팅' in x) or ('학회원' in x):
            return '내부 네트워크'
        if '지인' in x:
            return '지인'
        if ('대외활동' in x) or ('링커리어' in x) or ('캠퍼' in x):
            return '대외활동 플랫폼'
        if ('블라인드' in x) or ('링크드인' in x):
            return '직장인 커뮤니티'
        if ('행사' in x) or ('페스티벌' in x) or ('wave' in x.lower()):
            return '행사'
        if ('검색' in x) or ('블로그' in x):
            return '검색/블로그'
        return '기타'

    if 'inflow_route' in df.columns:
        df['inflow_route'] = df['inflow_route'].apply(route_group)

    # 참여 시간
    def time_band(x):
        # time_input이 NaN일 수 있으니 안전 처리
        if pd.isna(x):
            return 'missing_time'
        if x < 2:
            return 'low_time'
        elif 2 <= x <= 5:
            return 'optimal_time'
        else:
            return 'over_time'

    if 'time_input' in df.columns:
        df['time_band'] = df['time_input'].apply(time_band)

    # 전공 그룹화 (major1_1)
    def major_group(val):
        if pd.isna(val):
            return '미응답'
        x = str(val).strip().lower()

        if any(k in x for k in ['컴퓨터','소프트웨어','ai','데이터','정보','통신','전자','전기','공학','it','인공지능']):
            return 'IT(컴퓨터 공학 포함)'
        if any(k in x for k in ['경영','경제','금융','무역','통상','회계']):
            return '경영학'
        if any(k in x for k in ['수학','통계','물리','화학','생명','과학']):
            return '자연과학'
        if any(k in x for k in ['심리','사회','행정','정치','외교','언론','미디어']):
            return '사회과학'
        if any(k in x for k in ['영어','국어','문학','어학','철학','사학']):
            return '인문학'
        if '법' in x:
            return '법학'
        if any(k in x for k in ['의학','약학','보건','간호','의예']):
            return '의약학'
        if '교육' in x:
            return '교육학'
        if any(k in x for k in ['디자인','체육','음악','미술','예술','스포츠']):
            return '예체능'
        return '기타'

    if 'major1_1' in df.columns:
        df['major1_1'] = df['major1_1'].apply(major_group)

    # 이수학기
    if 'completed_semester' in df.columns:
        df['completed_semester'] = pd.to_numeric(df['completed_semester'], errors='coerce')
        df['completed_semester'] = df['completed_semester'].where(df['completed_semester'].between(1, 10))
        df['completed_semester'] = df['completed_semester'].fillna(0)

    # 자격증 리스트/개수
    def clean_cert(x):
        if isinstance(x, list):
            x = ','.join(x)
        x = str(x)

        invalid = {'없음', '기타', '준비중'}
        out = []
        for i in x.split(','):
            t = i.strip()
            if (t == '') or (t == 'nan'):
                continue
            if t in invalid:
                continue
            if t.startswith('준비중'):
                continue
            out.append(t)
        return out

    if 'certificate_acquisition' in df.columns:
        df['cert_list'] = df['certificate_acquisition'].apply(clean_cert)
        df['cert_cnt'] = df['cert_list'].apply(len)
    else:
        df['cert_cnt'] = 0

    # 학습 의지
    if 'onedayclass_topic' in df.columns:
        topics = df['onedayclass_topic'].fillna('').astype(str).str.strip()
        topics = (topics
                  .str.replace('데이터 시각화 (Matplotlib, Seaborn 등)', 'DATA_VIZ', regex=False)
                  .str.replace('머신러닝 / 딥러닝 응용', 'ML_DL', regex=False))

        topic_map = {
            'Python 응용': 'python',
            'SQL 응용': 'sql',
            '웹 크롤링': 'crawl',
            'DATA_VIZ': 'viz',
            'ML_DL': 'ml'
        }

        for raw, name in topic_map.items():
            df[f'topic_{name}'] = topics.str.contains(raw, regex=False).astype(int)

        df['topic_count'] = topics.apply(lambda x: 0 if x == '' else len([p.strip() for p in x.split(',') if p.strip()]))
        df['topic_count'] = df['topic_count'].clip(upper=5)
    else:
        for name in ['python','sql','crawl','viz','ml']:
            df[f'topic_{name}'] = 0
        df['topic_count'] = 0

    # 이전 참여 경험/횟수
    prev_cols = [c for c in df.columns if c.startswith('previous_class_')]
    if len(prev_cols) > 0:
        df['prev_class_cnt'] = (
            df[prev_cols]
            .apply(lambda col: col.astype(str).str.findall(r'000\d'))
            .applymap(len)
            .sum(axis=1)
            .astype(int)
        )
    else:
        df['prev_class_cnt'] = 0

    df['has_previous'] = (df['prev_class_cnt'] > 0).astype(int)

    # 현재 수강 과목 개수
    class_cols = [c for c in ['class1', 'class2', 'class3', 'class4'] if c in df.columns]
    if len(class_cols) > 0:
        df['class_cnt'] = df[class_cols].notna().sum(axis=1).astype(int)
    else:
        df['class_cnt'] = 0
    
    
    return df

# -----------------------------
# 2) train 타겟 이용 파생 (time_risk, job_cnt)
# -----------------------------
def target_dependent_features(df_train: pd.DataFrame, df_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df_train = df_train.copy()
    df_test = df_test.copy()

    # desired_job -> job_cnt
    # (A. B. C.) 형태에서 A~J 코드 찾기
    df_train['desired_job_list'] = (
        df_train['desired_job']
        .fillna("")
        .astype(str)
        .str.findall(r'\b([A-J])\.')
    )
    df_test['desired_job_list'] = (
        df_test['desired_job']
        .fillna("")
        .astype(str)
        .str.findall(r'\b([A-J])\.')
    )

    df_train['job_cnt'] = df_train['desired_job_list'].apply(len)
    df_test['job_cnt'] = df_test['desired_job_list'].apply(len)

    # time_risk: train에서 time_input별 완료율을 계산해서 매핑 (target=completed 필요)
    if ('time_input' in df_train.columns) and ('completed' in df_train.columns):
        summary = (
            df_train
            .groupby('time_input')['completed']
            .agg(['count', 'mean'])
            .rename(columns={'mean': 'completion_rate'})
        )
        global_rate = df_train['completed'].mean()

        df_train['time_risk'] = 1 - df_train['time_input'].map(summary['completion_rate']).fillna(global_rate)
        df_test['time_risk']  = 1 - df_test['time_input'].map(summary['completion_rate']).fillna(global_rate)
    else:
        df_train['time_risk'] = np.nan
        df_test['time_risk'] = np.nan

    # desired_job_list 제거
    df_train.drop(columns=['desired_job_list'], inplace=True, errors='ignore')
    df_test.drop(columns=['desired_job_list'], inplace=True, errors='ignore')
    
    
    return df_train, df_test

def added_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # 여기에 추가적인 파생변수 생성 코드를 작성하세요.
    #-------------------------추가---------------------------
    # 준비도 지표
    df['prep_score'] = df['cert_cnt'] + df['topic_count']
    
    # 지속성 지표
    df['class_ratio'] = df['prev_class_cnt'] / df['class_cnt'].replace(0, np.nan)
    
    #sqld&빅분기 관련 지표
    df['sqld_cert'] = df['cert_list'].apply(lambda x: 1 if 'SQLD' in x else 0)
    df['bigdata_cert'] = df['cert_list'].apply(lambda x: 1 if '빅데이터' in x else 0)
    df['else_cert'] = (df['cert_cnt'] - (df['sqld_cert'] + df['bigdata_cert'])) > 0
    
    # 가중치 부여한 자격증 개수
    df['cert_weighted'] = df['cert_cnt'] + df['bigdata_cert'] + df['sqld_cert']

    #offline 여부
    df['offline'] = df['hope_for_group'].map({
        '네. 오프라인으로 참여하고 싶어요': 1,
        '네. 온라인으로 참여하고 싶어요': 0
    }).fillna(-1)
    
    df['prep_score_bin'] = pd.qcut(
        df['prep_score'], q=5, labels=False, duplicates='drop'
    )

    df['cert_cnt_bin'] = pd.qcut(
        df['cert_cnt'], q=5, labels=False, duplicates='drop'
    )
    df['var1'] = df['prep_score'] * df['topic_count']
    df['var2'] = df['prep_score'] * df['cert_cnt']
    df['var3'] = df['job_cnt'] * df['prep_score']
    df['var4'] =  df['completed_semester'] / df['prep_score']
    df['var5'] = df['completed_semester'] / df['job_cnt'] 
    
    for col in ['var4','var5']:
        finite_vals = df.loc[np.isfinite(df[col]), col]
        max_val = finite_vals.max()
        df.loc[np.isinf(df[col]), col] = max_val * 1.1
    
    return df

df_train = make_features(df_train)
df_test  = make_features(df_test)

df_train, df_test = target_dependent_features(df_train, df_test)

df_train = added_features(df_train)
df_test  = added_features(df_test)



/var/folders/7x/ckht72xn7276hfpwcr1r7vwr0000gn/T/ipykernel_42979/1599426193.py:134: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(len)
/var/folders/7x/ckht72xn7276hfpwcr1r7vwr0000gn/T/ipykernel_42979/1599426193.py:134: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(len)


In [120]:
df_train.columns

Index(['school1', 'major type', 'major1_1', 'major1_2', 'major_data', 'job',
       'class1', 'class2', 'class3', 'class4', 're_registration',
       'contest_award', 'nationality', 'inflow_route', 'whyBDA',
       'what_to_gain', 'hope_for_group', 'previous_class_3',
       'previous_class_4', 'previous_class_5', 'previous_class_6',
       'previous_class_7', 'previous_class_8', 'major_field',
       'desired_career_path', 'completed_semester', 'project_type',
       'time_input', 'desired_job', 'certificate_acquisition',
       'desired_certificate', 'desired_job_except_data', 'incumbents_level',
       'incumbents_lecture', 'incumbents_company_level',
       'incumbents_lecture_type', 'incumbents_lecture_scale',
       'incumbents_lecture_scale_reason', 'interested_company',
       'expected_domain', 'contest_participation', 'idea_contest',
       'onedayclass_topic', 'completed', 'time_band', 'cert_list', 'cert_cnt',
       'topic_python', 'topic_sql', 'topic_crawl', 'topic_viz', '

# TabNet

In [6]:
TARGET = "completed"

drop_cols = [
    TARGET,
    "certificate_acquisition",
    "cert_list",
    "desired_job"   # 이미 파생함
]

FEATURES = [c for c in df_train.columns if c not in drop_cols]

X = df_train[FEATURES].copy()
y = df_train[TARGET].copy()

X_test = df_test[FEATURES].copy()


In [8]:
cat_cols = X.select_dtypes(include="object").columns.tolist()

cat_idxs = []
cat_dims = []

for col in cat_cols:

    # train 기준으로 카테고리 사전 생성
    train_vals = X[col].astype(str).unique().tolist()
    mapping = {v:i for i,v in enumerate(train_vals)}

    # unknown index 하나 추가
    unknown_idx = len(mapping)

    # 매핑 적용
    X[col] = X[col].astype(str).map(mapping).fillna(unknown_idx).astype(int)
    X_test[col] = X_test[col].astype(str).map(mapping).fillna(unknown_idx).astype(int)

    cat_idxs.append(X.columns.get_loc(col))
    cat_dims.append(len(mapping)+1)


In [9]:
X = X.fillna(-1)
X_test = X_test.fillna(-1)


### 모델 학습

In [10]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [12]:
import torch

clf = TabNetClassifier(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=3,  # 기본 무난

    n_d=16,
    n_a=16,
    n_steps=5,
    gamma=1.5,

    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),

    scheduler_params={"step_size":50, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,

    verbose=1
)


/opt/miniconda3/lib/python3.13/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [17]:
clf.fit(
    X_train.values, y_train.values,
    eval_set=[(X_valid.values, y_valid.values)],
    eval_metric=["auc"],
    max_epochs=50
)


epoch 0  | loss: 0.0     | val_0_auc: 0.56222 |  0:00:00s
epoch 1  | loss: 0.0     | val_0_auc: 0.56222 |  0:00:00s
epoch 2  | loss: 0.0     | val_0_auc: 0.56222 |  0:00:00s
epoch 3  | loss: 0.0     | val_0_auc: 0.56222 |  0:00:00s
epoch 4  | loss: 0.0     | val_0_auc: 0.56222 |  0:00:00s
epoch 5  | loss: 0.0     | val_0_auc: 0.56222 |  0:00:00s
epoch 6  | loss: 0.0     | val_0_auc: 0.56222 |  0:00:00s
epoch 7  | loss: 0.0     | val_0_auc: 0.56222 |  0:00:00s
epoch 8  | loss: 0.0     | val_0_auc: 0.56222 |  0:00:00s
epoch 9  | loss: 0.0     | val_0_auc: 0.56222 |  0:00:00s
epoch 10 | loss: 0.0     | val_0_auc: 0.56222 |  0:00:00s

Early stopping occurred at epoch 10 with best_epoch = 0 and best_val_0_auc = 0.56222


/opt/miniconda3/lib/python3.13/site-packages/torch/optim/lr_scheduler.py:182: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found object

In [16]:
pred_val = clf.predict_proba(X_valid.values)[:,1]
auc = roc_auc_score(y_valid, pred_val)
print("VALID AUC:", auc)


AttributeError: 'TabNetClassifier' object has no attribute 'network'

In [ ]:
pred_test = clf.predict_proba(X_test.values)[:,1]
